In [2]:
import ee
import pandas as pd
import time

# ==========================================
# STEP 1: INITIALIZE GOOGLE EARTH ENGINE
# ==========================================
PROJECT_ID = "ecoaudit-ai-498509"

try:
    print(f"Attempting to initialize Earth Engine with project: {PROJECT_ID}...")
    ee.Initialize(project=PROJECT_ID)
    print("🎉 Success! Earth Engine initialized perfectly.")
except Exception as e:
    print("Authentication credentials missing or expired. Initiating sign-in link...")
    ee.Authenticate()
    ee.Initialize(project=PROJECT_ID)
    print("🎉 Success! Earth Engine initialized after re-authentication.")

# ==========================================
# STEP 2: DEFINE YOUR GLOBAL DATA POINTS
# ==========================================
# Copy and paste this expanded global baseline matrix into Member 2's Step 2:
global_coordinates_pool = [
    # --- 1. TROPICAL RAINFORESTS (Maximum Carbon/Biomass Sinks) ---
    {"lat": -3.4653, "lon": -62.2159, "agb": 320.5},   # Amazon Basin, Brazil
    {"lat": 1.2921, "lon": 114.3211, "agb": 295.0},    # Borneo Canopy, Indonesia
    {"lat": -1.2541, "lon": 23.4215, "agb": 280.0},    # Congo Basin, Central Africa
    {"lat": 11.2341, "lon": 76.1245, "agb": 195.4},    # Western Ghats Reserves, India
    {"lat": -18.1562, "lon": 142.3642, "agb": 150.0},  # Queensland Tropical Region, Australia

    # --- 2. TEMPERATE & BOREAL FORESTS (Medium to High Carbon) ---
    {"lat": 45.4215, "lon": -75.6972, "agb": 115.2},   # Algonquin Woodlands, Canada
    {"lat": 48.1351, "lon": 11.5820, "agb": 98.7},     # Bavarian Black Forest, Germany
    {"lat": 61.2181, "lon": -149.9003, "agb": 82.1},   # Boreal Taiga Timberland, Alaska
    {"lat": 32.7767, "lon": -96.7970, "agb": 45.3},    # Southern Pine Woodlands, Texas
    {"lat": 55.7558, "lon": 37.6173, "agb": 70.4},     # Mixed Siberian Woodland Margins

    # --- 3. ARID SAVANNAS & SHRUBLANDS (Low/Seasonal Carbon) ---
    {"lat": -22.9576, "lon": 18.4905, "agb": 12.4},    # Kalahari Bushlands, Namibia
    {"lat": 9.0820, "lon": 8.6753, "agb": 18.5},      # Nigerian Guinean Savanna
    {"lat": 26.2006, "lon": 92.9376, "agb": 35.8},     # Assam Grasslands, India

    # --- 4. BARREN DESERTS (True 0.0 Carbon Baseline) ---
    {"lat": 23.3548, "lon": 69.6692, "agb": 0.0},      # Rann of Kutch Salt Flats, India
    {"lat": 24.8607, "lon": 67.0011, "agb": 0.0},      # Thar Desert Sand Dunes
    {"lat": 23.8859, "lon": 11.2841, "agb": 0.0},      # Sahara Desert Core, Algeria
    {"lat": 36.5323, "lon": -116.9325, "agb": 0.0},    # Death Valley Basin, USA
    {"lat": -24.3768, "lon": -69.2134, "agb": 0.0},    # Atacama Hyper-Arid Plateau, Chile
    {"lat": 22.0144, "lon": 44.7411, "agb": 0.0},      # Rub' al Khali (Empty Quarter), Saudi Arabia

    # --- 5. URBAN MEGACITIES (True 0.0 Carbon Baseline) ---
    {"lat": 27.1751, "lon": 78.0421, "agb": 0.0},      # Agra Urban Concrete, India
    {"lat": 13.0827, "lon": 80.2707, "agb": 0.0},      # Chennai Metropolitan Core, India
    {"lat": 19.0760, "lon": 72.8777, "agb": 0.0},      # Downtown Mumbai Commercial Grid, India
    {"lat": 28.6139, "lon": 77.2090, "agb": 0.0},      # New Delhi High-Density Zone, India
    {"lat": 40.7128, "lon": -74.0060, "agb": 0.0},     # Manhattan Skyscraper Grid, New York
    {"lat": 35.6762, "lon": 139.6503, "agb": 0.0},     # Tokyo City Asphalt Layout, Japan
    {"lat": 51.5074, "lon": -0.1278, "agb": 0.0},      # Greater London Built Environment, UK
    {"lat": -23.5505, "lon": -46.6333, "agb": 0.0},    # São Paulo Concrete Jungle, Brazil

    # --- 6. GLACIERS & ICE CAPS (True 0.0 Carbon Baseline) ---
    {"lat": 64.9631, "lon": -19.0208, "agb": 0.0},     # Vatnajökull Glacial Ice Shield, Iceland
    {"lat": 74.2764, "lon": -40.3251, "agb": 0.0},     # Greenland Inland Ice Sheet
    {"lat": -75.2509, "lon": -0.0713, "agb": 0.0}      # Antarctic Coastal Ice Shelf
]

# ==========================================
# STEP 3: ENVIRONMENT FEATURE EXTRACTION CORES
# ==========================================
def extract_environmental_covariates(lat, lon, true_biomass_label):
    point = ee.Geometry.Point([lon, lat])

    # 1. Fetch Harmonized Sentinel-2 Surface Reflectance
    s2_collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
        .filterBounds(point) \
        .filterDate('2025-01-01', '2025-12-31') \
        .sort('CLOUDY_PIXEL_PERCENTAGE')

    s2_img = s2_collection.first()
    if s2_img is None:
        raise ValueError("No clear Sentinel-2 satellite tiles found for this coordinate.")

    # Calculate Custom Indices expected by the ML model
    ndvi = s2_img.normalizedDifference(['B8', 'B4']).rename('NDVI')
    evi = s2_img.expression('2.5 * ((N - R) / (N + 6 * R - 7.5 * B + 1))', {
        'N': s2_img.select('B8'), 'R': s2_img.select('B4'), 'B': s2_img.select('B2')
    }).rename('EVI')
    ndre = s2_img.normalizedDifference(['B8', 'B5']).rename('NDRE')

    # 2. Fetch Sentinel-1 SAR Radar Data
    s1_img = ee.ImageCollection('COPERNICUS/S1_GRD') \
        .filterBounds(point) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV')) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH')) \
        .first()

    # 3. Fetch Topography (SRTM) and Bioclimatic layers (WorldClim FIXED)
    worldclim = ee.ImageCollection('WORLDCLIM/V1/MONTHLY').mean()
    srtm = ee.Image('USGS/SRTMGL1_003')

    # Reduce Spatial Layers to Point Mean Dicts
    s2_vals = s2_img.select(['B2', 'B4', 'B5', 'B6', 'B7', 'B8']).reduceRegion(ee.Reducer.mean(), point, 10).getInfo()

    if s1_img:
        s1_vals = s1_img.select(['VV', 'VH']).reduceRegion(ee.Reducer.mean(), point, 10).getInfo()
    else:
        s1_vals = {'VV': -10.0, 'VH': -17.0}

    temp = worldclim.select('tavg').reduceRegion(ee.Reducer.mean(), point, 30).get('tavg').getInfo()
    precip = worldclim.select('prec').reduceRegion(ee.Reducer.mean(), point, 30).get('prec').getInfo()
    elev = srtm.reduceRegion(ee.Reducer.mean(), point, 30).get('elevation').getInfo()

    ndvi_val = ndvi.reduceRegion(ee.Reducer.mean(), point, 10).getInfo().get('NDVI')
    evi_val = evi.reduceRegion(ee.Reducer.mean(), point, 10).getInfo().get('EVI')
    ndre_val = ndre.reduceRegion(ee.Reducer.mean(), point, 10).getInfo().get('NDRE')

    return {
        'B2': s2_vals.get('B2'), 'B4': s2_vals.get('B4'), 'B5': s2_vals.get('B5'),
        'B6': s2_vals.get('B6'), 'B7': s2_vals.get('B7'), 'B8': s2_vals.get('B8'),
        'VV': s1_vals.get('VV'), 'VH': s1_vals.get('VH'),
        'climate_temp': temp,
        'climate_precip': (precip * 12) if precip else 0,
        'terrain_elevation': elev if elev else 0,
        'NDVI': ndvi_val, 'EVI': evi_val, 'NDRE': ndre_val,
        'TARGET_AGB': true_biomass_label
    }

# ==========================================
# STEP 4: RUN AUTOMATED PIPELINE LOOP
# ==========================================
dataset_rows = []
print("\nBeginning structural multi-biome raster extraction pipeline...")

for idx, spot in enumerate(global_coordinates_pool):
    try:
        print(f"Querying planetary telemetry for Point {idx+1}/{len(global_coordinates_pool)} [Lat: {spot['lat']}, Lon: {spot['lon']}]...")
        row_data = extract_environmental_covariates(spot['lat'], spot['lon'], spot['agb'])
        dataset_rows.append(row_data)
        time.sleep(0.5)
    except Exception as e:
        print(f"⚠️ Flagged point index {idx+1} (Skipped due to cloud filtering or metadata omission): {e}")

# ==========================================
# STEP 5: CLEAN AND SAVE THE CSV MATRIX
# ==========================================
if dataset_rows:
    df_global = pd.DataFrame(dataset_rows).dropna()
    df_global.to_csv("cleaned_training_data.csv", index=False)
    print("\n=======================================================")
    print(f"🎯 ANALYSIS COMPLETE: {df_global.shape[0]} data rows packed successfully!")
    print("Download 'cleaned_training_data.csv' from your Colab files sidebar and pass it to Member 3.")
    print("=======================================================")
else:
    print("\n❌ CRITICAL: No records were extracted. Verify your network or coordinate selections.")

Attempting to initialize Earth Engine with project: ecoaudit-ai-498509...
Authentication credentials missing or expired. Initiating sign-in link...
🎉 Success! Earth Engine initialized after re-authentication.

Beginning structural multi-biome raster extraction pipeline...
Querying planetary telemetry for Point 1/30 [Lat: -3.4653, Lon: -62.2159]...
Querying planetary telemetry for Point 2/30 [Lat: 1.2921, Lon: 114.3211]...
Querying planetary telemetry for Point 3/30 [Lat: -1.2541, Lon: 23.4215]...
Querying planetary telemetry for Point 4/30 [Lat: 11.2341, Lon: 76.1245]...
Querying planetary telemetry for Point 5/30 [Lat: -18.1562, Lon: 142.3642]...
Querying planetary telemetry for Point 6/30 [Lat: 45.4215, Lon: -75.6972]...
Querying planetary telemetry for Point 7/30 [Lat: 48.1351, Lon: 11.582]...
Querying planetary telemetry for Point 8/30 [Lat: 61.2181, Lon: -149.9003]...
Querying planetary telemetry for Point 9/30 [Lat: 32.7767, Lon: -96.797]...
Querying planetary telemetry for Point